# 🧠 FinScope AI — Advanced Analytics Notebook
> Real Supabase data → ML models → Beautiful visualizations → AI insights

---
**Sections:**
1. 🔌 Setup & Connect
2. 📊 Load & Explore Data
3. 🎨 Dashboard-Style Visualizations
4. 🔵 K-Means Spending Cluster Analysis
5. 🌡️ Heatmap & Correlation Matrix
6. 📈 Time-Series Decomposition
7. 🤖 Anomaly Detection (Isolation Forest)
8. 🔮 Spending Forecast (ARIMA-style)
9. 🧬 User Behavior DNA
10. 💾 Export Results

## 🔌 1. Setup & Connect

In [1]:
# ─── Install dependencies (run once) ─────────────────────────────────────────
# !pip install supabase scikit-learn pandas numpy matplotlib seaborn plotly

import warnings
warnings.filterwarnings('ignore')

import json, os, re
from datetime import datetime, timedelta
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

# ML
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Supabase
from supabase import create_client, Client
from dotenv import load_dotenv

# ─── FinScope Dark Theme ──────────────────────────────────────────────────────
BG       = '#0f1729'
PANEL    = '#1a2340'
BORDER   = '#1e2a4a'
BLUE     = '#3b82f6'
CYAN     = '#06b6d4'
PURPLE   = '#8b5cf6'
VIOLET   = '#a78bfa'
GREEN    = '#10b981'
RED      = '#ef4444'
YELLOW   = '#f59e0b'
PINK     = '#ec4899'
ORANGE   = '#f97316'
GREY     = '#6b7280'
TEXT     = '#94a3b8'
WHITE    = '#ffffff'

CAT_COLORS = {
    'Groceries':     BLUE,
    'Subscriptions': PURPLE,
    'Shopping':      VIOLET,
    'Transport':     CYAN,
    'Dining':        YELLOW,
    'Bills':         RED,
    'Health':        GREEN,
    'Entertainment': PINK,
    'Education':     ORANGE,
    'Other':         GREY,
    'Income':        GREEN,
}

def finscope_style(ax, fig, title='', subtitle=''):
    """Apply FinScope dark theme to a matplotlib figure."""
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    ax.tick_params(colors=TEXT, labelsize=9)
    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)
    for spine in ax.spines.values():
        spine.set_color(BORDER)
    ax.grid(color=BORDER, linewidth=0.5, alpha=0.5)
    if title:
        ax.set_title(title, color=WHITE, fontsize=13, fontweight='bold', pad=10)
    if subtitle:
        ax.text(0.5, 1.025, subtitle, transform=ax.transAxes,
                ha='center', fontsize=9, color=TEXT, style='italic')

FS_CMAP = LinearSegmentedColormap.from_list(
    'finscope', [BG, BORDER, BLUE, CYAN, WHITE], N=256
)

print('✅ All imports successful')
print(f'   📦 pandas {pd.__version__} | scikit-learn | matplotlib | seaborn')
print('   🎨 FinScope dark theme ready')

✅ All imports successful
   📦 pandas 2.3.3 | scikit-learn | matplotlib | seaborn
   🎨 FinScope dark theme ready


In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

SUPABASE_URL      = os.getenv('SUPABASE_URL')
SUPABASE_KEY      = os.getenv('SUPABASE_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

# ── Optional Supabase connection ──────────────────────────────────────────────
supabase = None
if SUPABASE_URL and SUPABASE_KEY:
    try:
        from supabase import create_client, Client
        supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
        print(f'✅ Connected to Supabase: {SUPABASE_URL}')
    except Exception as e:
        print(f'⚠️  Supabase connection failed: {e}')
        print('   → Falling back to synthetic data')
else:
    print('ℹ️  No Supabase credentials found — using synthetic data')

ℹ️  No Supabase credentials found — using synthetic data


## 📊 2. Load & Explore Data

In [7]:
# ─── Load all transactions ────────────────────────────────────────────────────
def load_transactions(user_id=None, limit=5000):
    query = supabase.table('user_transactions').select('*').limit(limit)
    if user_id:
        query = query.eq('user_id', user_id)
    result = query.order('created_at', desc=True).execute()
    if not result.data:
        return pd.DataFrame()

    df = pd.DataFrame(result.data)
    df['created_at']   = pd.to_datetime(df['created_at'])
    df['amount']       = pd.to_numeric(df['amount'], errors='coerce')
    df['month']        = df['created_at'].dt.to_period('M')
    df['week']         = df['created_at'].dt.isocalendar().week
    df['day_of_week']  = df['created_at'].dt.day_name()
    df['day_num']      = df['created_at'].dt.dayofweek   # 0=Mon
    df['hour']         = df['created_at'].dt.hour
    df['date']         = df['created_at'].dt.date
    df['day_of_month'] = df['created_at'].dt.day
    df['is_weekend']   = df['day_num'].isin([5, 6])
    return df

# ─── Generate synthetic data if DB is empty ───────────────────────────────────
def generate_synthetic(n_months=6, n_users=3, seed=42):
    np.random.seed(seed)
    cats = {
        'Subscriptions': [('Spotify', 9.99),   ('Netflix', 15.99), ('Disney+', 13.99), ('YouTube', 13.99)],
        'Dining':        [('Starbucks', 6.50),  ('Uber Eats', 28.0), ('Restaurant', 45.0), ('Greggs', 4.50)],
        'Transport':     [('Uber', 18.0),        ('Gas', 55.0),       ('Parking', 12.0),   ('TfL', 8.0)],
        'Shopping':      [('Amazon', 35.0),      ('Nike', 90.0),      ('Zara', 60.0),      ('eBay', 28.0)],
        'Bills':         [('Electricity', 95.0), ('Internet', 60.0),  ('Phone', 45.0),     ('Water', 30.0)],
        'Groceries':     [('Tesco', 75.0),       ('Waitrose', 120.0), ('Aldi', 45.0),      ('Lidl', 38.0)],
        'Health':        [('Gym', 40.0),         ('Pharmacy', 15.0),  ('Doctor', 50.0)],
        'Entertainment': [('Cinema', 18.0),      ('Steam', 25.0),     ('Concert', 65.0)],
        'Education':     [('Udemy', 19.99),      ('Books', 22.0),     ('Course', 49.0)],
    }
    rows, now = [], datetime.now()
    user_profiles = [
        {'multiplier': 1.0, 'bias': {}},
        {'multiplier': 1.6, 'bias': {'Dining': 2.5, 'Entertainment': 2.0}},
        {'multiplier': 0.7, 'bias': {'Groceries': 1.8, 'Bills': 1.4}},
    ]
    for u_idx in range(n_users):
        uid  = f'user-{u_idx:03d}'
        prof = user_profiles[u_idx % len(user_profiles)]
        for m in range(n_months):
            mstart = now - timedelta(days=30 * (n_months - m))
            rows.append({'user_id': uid, 'name': 'Salary',
                         'amount': round(np.random.uniform(2800, 5500), 2),
                         'type': 'deposit', 'category': 'Income',
                         'created_at': (mstart + timedelta(days=1)).isoformat()})
            for cat, items in cats.items():
                bias = prof['bias'].get(cat, 1.0)
                n_txns = int(np.random.randint(1, 6) * bias)
                for _ in range(n_txns):
                    item, base = items[np.random.randint(len(items))]
                    amt = round(base * prof['multiplier'] * np.random.uniform(0.75, 1.5), 2)
                    # Inject anomaly ~3% chance
                    if np.random.random() < 0.03:
                        amt = round(amt * np.random.uniform(5, 12), 2)
                    day = np.random.randint(0, 29)
                    rows.append({'user_id': uid, 'name': item, 'amount': amt,
                                 'type': 'expense', 'category': cat,
                                 'created_at': (mstart + timedelta(days=day,
                                     hours=np.random.randint(7,23),
                                     minutes=np.random.randint(0,60))).isoformat()})
    df = pd.DataFrame(rows)
    df['created_at']   = pd.to_datetime(df['created_at'])
    df['amount']       = pd.to_numeric(df['amount'])
    df['month']        = df['created_at'].dt.to_period('M')
    df['week']         = df['created_at'].dt.isocalendar().week
    df['day_of_week']  = df['created_at'].dt.day_name()
    df['day_num']      = df['created_at'].dt.dayofweek
    df['hour']         = df['created_at'].dt.hour
    df['date']         = df['created_at'].dt.date
    df['day_of_month'] = df['created_at'].dt.day
    df['is_weekend']   = df['day_num'].isin([5, 6])
    return df


# ─── Load (or synthesize) ─────────────────────────────────────────────────────
df = load_transactions()
if df.empty:
    print('⚠️  No Supabase data found — using synthetic dataset for demo')
    df = generate_synthetic(n_months=6, n_users=3)
else:
    print(f'✅ Loaded {len(df)} real transactions from Supabase')

expenses  = df[df['type'] == 'expense'].copy()
deposits  = df[df['type'] == 'deposit'].copy()

print(f'\n📦 Dataset overview')
print(f'   Rows         : {len(df):,}')
print(f'   Users        : {df["user_id"].nunique()}')
print(f'   Expenses     : {len(expenses):,}  (${expenses["amount"].sum():,.2f} total)')
print(f'   Deposits     : {len(deposits):,}')
print(f'   Date range   : {df["created_at"].min().date()} → {df["created_at"].max().date()}')
print(f'   Categories   : {expenses["category"].nunique()}')

AttributeError: 'NoneType' object has no attribute 'table'

## 🎨 3. Dashboard-Style Visualizations

In [ ]:
# ─── Fig 1: FinScope Dashboard Overview (4-panel) ────────────────────────────
fig = plt.figure(figsize=(18, 10), facecolor=BG)
fig.suptitle('FinScope AI — Financial Dashboard', color=WHITE,
             fontsize=18, fontweight='bold', y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32,
                       top=0.92, bottom=0.08, left=0.07, right=0.97)

# ── Panel A: Category Donut ────────────────────────────────────────────────
ax_donut = fig.add_subplot(gs[0, 0])
cat_totals = expenses.groupby('category')['amount'].sum().sort_values(ascending=False)
colors_donut = [CAT_COLORS.get(c, GREY) for c in cat_totals.index]
wedges, texts, autos = ax_donut.pie(
    cat_totals.values, labels=None,
    colors=colors_donut, autopct='%1.0f%%',
    pctdistance=0.78, startangle=90,
    wedgeprops=dict(width=0.52, edgecolor=BG, linewidth=2),
)
for auto in autos:
    auto.set_color(WHITE); auto.set_fontsize(8); auto.set_fontweight('bold')
ax_donut.set_facecolor(BG)
ax_donut.set_title('Spending by Category', color=WHITE, fontsize=12, fontweight='bold', pad=10)
legend_patches = [mpatches.Patch(color=CAT_COLORS.get(c, GREY), label=c)
                  for c in cat_totals.index]
ax_donut.legend(handles=legend_patches, loc='center left', bbox_to_anchor=(1.05, 0.5),
                fontsize=8, labelcolor=TEXT, facecolor=PANEL, edgecolor=BORDER, framealpha=0.9)
circle = plt.Circle((0, 0), 0.35, color=BG)
ax_donut.add_patch(circle)
ax_donut.text(0, 0, f'${cat_totals.sum():,.0f}', ha='center', va='center',
              color=WHITE, fontsize=11, fontweight='bold')

# ── Panel B: Monthly Trend Line ────────────────────────────────────────────
ax_trend = fig.add_subplot(gs[0, 1])
ax_trend.set_facecolor(BG)
monthly = expenses.groupby('month')['amount'].sum().reset_index()
monthly['month_str'] = monthly['month'].astype(str)
x = range(len(monthly))
ax_trend.fill_between(x, monthly['amount'], alpha=0.18, color=BLUE)
ax_trend.plot(x, monthly['amount'], color=BLUE, linewidth=2.5,
              marker='o', markersize=7, markerfacecolor=WHITE, markeredgecolor=BLUE, markeredgewidth=2)
for xi, row in zip(x, monthly.itertuples()):
    ax_trend.annotate(f'${row.amount:,.0f}', (xi, row.amount),
                      textcoords='offset points', xytext=(0, 11),
                      ha='center', color=TEXT, fontsize=8, fontweight='bold')
ax_trend.set_xticks(list(x))
ax_trend.set_xticklabels(monthly['month_str'], rotation=35, ha='right', color=TEXT, fontsize=8)
ax_trend.set_ylabel('Expenses ($)', color=TEXT, fontsize=9)
finscope_style(ax_trend, fig, 'Monthly Spending Trend')

# ── Panel C: Top 8 Categories (horizontal bar) ────────────────────────────
ax_bar = fig.add_subplot(gs[0, 2])
ax_bar.set_facecolor(BG)
top8 = cat_totals.head(8).sort_values()
colors_bar = [CAT_COLORS.get(c, GREY) for c in top8.index]
bars = ax_bar.barh(top8.index, top8.values, color=colors_bar, height=0.6, edgecolor='none')
for bar, val in zip(bars, top8.values):
    ax_bar.text(bar.get_width() + top8.max() * 0.02, bar.get_y() + bar.get_height() / 2,
                f'${val:,.0f}', va='center', color=WHITE, fontsize=8, fontweight='bold')
ax_bar.set_xlabel('Total Spent ($)', color=TEXT, fontsize=9)
finscope_style(ax_bar, fig, 'Top Spending Categories')
ax_bar.set_xlim(right=top8.max() * 1.25)
ax_bar.tick_params(colors=TEXT, labelsize=8)

# ── Panel D: Weekday vs Weekend (box plot) ─────────────────────────────────
ax_box = fig.add_subplot(gs[1, 0])
ax_box.set_facecolor(BG)
wd_data  = expenses[~expenses['is_weekend']]['amount']
we_data  = expenses[expenses['is_weekend']]['amount']
bp = ax_box.boxplot([wd_data, we_data], patch_artist=True,
                    medianprops=dict(color=WHITE, linewidth=2),
                    whiskerprops=dict(color=TEXT), capprops=dict(color=TEXT),
                    flierprops=dict(marker='o', color=RED, alpha=0.4, markersize=4))
bp['boxes'][0].set_facecolor(BLUE + '44'); bp['boxes'][0].set_edgecolor(BLUE)
bp['boxes'][1].set_facecolor(PURPLE + '44'); bp['boxes'][1].set_edgecolor(PURPLE)
ax_box.set_xticklabels(['Weekday', 'Weekend'], color=TEXT, fontsize=10)
ax_box.set_ylabel('Transaction Amount ($)', color=TEXT, fontsize=9)
finscope_style(ax_box, fig, 'Weekday vs Weekend Spending')

# ── Panel E: Day-of-Week Avg Spend ─────────────────────────────────────────
ax_dow = fig.add_subplot(gs[1, 1])
ax_dow.set_facecolor(BG)
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_avg = expenses.groupby('day_of_week')['amount'].mean().reindex(day_order)
dow_colors = [RED if d in ['Saturday','Sunday'] else BLUE for d in day_order]
ax_dow.bar(range(7), dow_avg.values, color=dow_colors, width=0.65, edgecolor='none')
ax_dow.set_xticks(range(7))
ax_dow.set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], color=TEXT, fontsize=9)
ax_dow.set_ylabel('Avg Transaction ($)', color=TEXT, fontsize=9)
finscope_style(ax_dow, fig, 'Avg Spend by Day of Week')
for i, val in enumerate(dow_avg.values):
    ax_dow.text(i, val + dow_avg.max() * 0.02, f'${val:.0f}',
                ha='center', color=WHITE, fontsize=8, fontweight='bold')

# ── Panel F: Hourly Spending Distribution ──────────────────────────────────
ax_hour = fig.add_subplot(gs[1, 2])
ax_hour.set_facecolor(BG)
hour_avg = expenses.groupby('hour')['amount'].sum()
all_hours = pd.Series(0, index=range(24))
all_hours.update(hour_avg)
grad = plt.cm.cool(np.linspace(0.1, 0.9, 24))
ax_hour.bar(range(24), all_hours.values, color=grad, width=0.85, edgecolor='none')
ax_hour.set_xlabel('Hour of Day', color=TEXT, fontsize=9)
ax_hour.set_ylabel('Total Spent ($)', color=TEXT, fontsize=9)
finscope_style(ax_hour, fig, 'Spending by Hour of Day')
ax_hour.set_xticks(range(0, 24, 3))
ax_hour.set_xticklabels([f'{h:02d}:00' for h in range(0,24,3)], color=TEXT, fontsize=8)

plt.savefig('finscope_dashboard_overview.png', dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()
print('✅ Saved: finscope_dashboard_overview.png')

## 🔵 4. K-Means Spending Cluster Analysis

In [ ]:
# ─── K-Means: Find optimal K (Elbow + Silhouette) ────────────────────────────
print('🔵 Running K-Means Cluster Analysis...')

# ── Feature engineering for clustering ──────────────────────────────────────
# Per-user-month spending fingerprint
le = LabelEncoder()
expenses['cat_num'] = le.fit_transform(expenses['category'])

# Build feature matrix: each row = one user's aggregated spending vector
pivot = expenses.groupby(['user_id', 'category'])['amount'].sum().unstack(fill_value=0)
pivot['total']      = pivot.sum(axis=1)
pivot['n_txns']     = expenses.groupby('user_id').size()
pivot['avg_txn']    = pivot['total'] / pivot['n_txns'].replace(0, np.nan)
pivot['max_txn']    = expenses.groupby('user_id')['amount'].max()
pivot['std_txn']    = expenses.groupby('user_id')['amount'].std().fillna(0)
pivot['weekend_pct']= (expenses[expenses['is_weekend']].groupby('user_id').size() /
                       expenses.groupby('user_id').size()).fillna(0)
pivot = pivot.fillna(0)

# If only 1 user, pivot per user×month instead
if len(pivot) < 3:
    pivot = expenses.groupby(['user_id', 'month', 'category'])['amount'].sum().unstack(fill_value=0)
    pivot['total']   = pivot.sum(axis=1)
    pivot['n_txns']  = expenses.groupby(['user_id','month']).size()
    pivot['avg_txn'] = pivot['total'] / pivot['n_txns'].replace(0, np.nan)
    pivot = pivot.fillna(0)
    pivot.index = [f'{str(u)[:8]}…|{str(m)}' for u, m in pivot.index]

X_raw = pivot.values
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# Determine max K
K_max = min(8, len(X) - 1)
K_range = range(2, K_max + 1)

inertias, sils = [], []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=12)
    km.fit(X)
    inertias.append(km.inertia_)
    if k < len(X):
        sils.append(silhouette_score(X, km.labels_))
    else:
        sils.append(0)

# ── Plot Elbow + Silhouette ──────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)

ax1.set_facecolor(BG)
ax1.plot(list(K_range), inertias, color=BLUE, linewidth=2.5,
         marker='o', markersize=8, markerfacecolor=WHITE, markeredgecolor=BLUE, markeredgewidth=2)
ax1.fill_between(list(K_range), inertias, alpha=0.12, color=BLUE)
ax1.set_xlabel('Number of Clusters (K)', color=TEXT); ax1.set_ylabel('Inertia (WCSS)', color=TEXT)
finscope_style(ax1, fig, 'Elbow Method — Find Optimal K')

ax2.set_facecolor(BG)
best_k = int(list(K_range)[np.argmax(sils)])
bar_colors = [CYAN if k == best_k else PURPLE for k in K_range]
ax2.bar(list(K_range), sils, color=bar_colors, width=0.65, edgecolor='none')
ax2.axhline(max(sils), color=CYAN, linestyle='--', linewidth=1.5, alpha=0.6)
ax2.text(K_range[-1], max(sils) * 1.02, f' Best K={best_k}', color=CYAN, fontsize=10, fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)', color=TEXT); ax2.set_ylabel('Silhouette Score', color=TEXT)
finscope_style(ax2, fig, 'Silhouette Score per K')

plt.tight_layout(pad=3)
plt.savefig('finscope_kmeans_selection.png', dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'✅ Optimal K = {best_k}  (Silhouette = {max(sils):.3f})')

In [ ]:
# ─── Final K-Means fit + PCA visualization ───────────────────────────────────
K = best_k
km_final = KMeans(n_clusters=K, random_state=42, n_init=20, max_iter=500)
labels = km_final.fit_predict(X)

pivot['cluster'] = labels
cluster_palette = [BLUE, CYAN, PURPLE, VIOLET, GREEN, RED, YELLOW, PINK]

# ── PCA for 2D visualization ─────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
var_explained = pca.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=BG)
fig.suptitle('K-Means Spending Cluster Analysis', color=WHITE, fontsize=15, fontweight='bold')

# ── Sub-plot 1: PCA Scatter ───────────────────────────────────────────────
ax = axes[0]; ax.set_facecolor(BG)
for c in range(K):
    mask = labels == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               color=cluster_palette[c % len(cluster_palette)],
               s=120, alpha=0.85, edgecolors=BG, linewidth=1.2, label=f'Cluster {c+1}')
centers_pca = pca.transform(km_final.cluster_centers_)
ax.scatter(centers_pca[:, 0], centers_pca[:, 1],
           color=WHITE, s=220, marker='★', zorder=10, edgecolors=YELLOW, linewidth=1.5)
for idx, row in enumerate(pivot.index):
    ax.annotate(str(row)[:10], (X_pca[idx, 0], X_pca[idx, 1]),
                textcoords='offset points', xytext=(5, 3),
                color=TEXT, fontsize=6.5, alpha=0.8)
ax.set_xlabel(f'PC1 ({var_explained[0]:.1f}% variance)', color=TEXT, fontsize=9)
ax.set_ylabel(f'PC2 ({var_explained[1]:.1f}% variance)', color=TEXT, fontsize=9)
finscope_style(ax, fig, 'PCA Cluster Scatter')
ax.legend(fontsize=8, facecolor=PANEL, edgecolor=BORDER, labelcolor=WHITE)

# ── Sub-plot 2: Cluster Avg Spend by Category (heatmap) ──────────────────
ax2 = axes[1]; ax2.set_facecolor(BG)
cat_cols = [c for c in pivot.columns if c in expenses['category'].unique()]
cluster_means = pivot.groupby('cluster')[cat_cols].mean()
im = ax2.imshow(cluster_means.T.values, cmap=FS_CMAP, aspect='auto')
ax2.set_xticks(range(K))
ax2.set_xticklabels([f'C{i+1}' for i in range(K)], color=TEXT, fontsize=10)
ax2.set_yticks(range(len(cat_cols)))
ax2.set_yticklabels(cat_cols, color=TEXT, fontsize=9)
for i in range(K):
    for j in range(len(cat_cols)):
        val = cluster_means.T.values[j, i]
        ax2.text(i, j, f'${val:.0f}', ha='center', va='center',
                 color=WHITE if val > cluster_means.values.mean() else TEXT, fontsize=7.5, fontweight='bold')
plt.colorbar(im, ax=ax2, shrink=0.7).ax.yaxis.set_tick_params(color=TEXT, labelcolor=TEXT)
finscope_style(ax2, fig, 'Cluster Avg Spend by Category ($)')

# ── Sub-plot 3: Cluster Size + Avg Total ─────────────────────────────────
ax3 = axes[2]; ax3.set_facecolor(BG)
cluster_summary = pivot.groupby('cluster').agg(
    count=('total', 'count'),
    avg_total=('total', 'mean'),
    avg_txns=('n_txns', 'mean') if 'n_txns' in pivot.columns else ('total', 'mean')
)
cx = np.arange(K)
w = 0.35
bars1 = ax3.bar(cx - w/2, cluster_summary['count'].values,
                width=w, color=[cluster_palette[i % len(cluster_palette)] for i in range(K)],
                edgecolor='none', label='# Members')
ax3_r = ax3.twinx()
ax3_r.set_facecolor(BG)
ax3_r.tick_params(colors=TEXT, labelsize=8)
bars2 = ax3_r.bar(cx + w/2, cluster_summary['avg_total'].values,
                  width=w, color=CYAN + '88', edgecolor=CYAN, linewidth=1.2, label='Avg Total ($)')
ax3.set_xticks(cx)
ax3.set_xticklabels([f'Cluster {i+1}' for i in range(K)], color=TEXT, fontsize=9)
ax3.set_ylabel('# Users/Months', color=TEXT, fontsize=9)
ax3_r.set_ylabel('Avg Total Spent ($)', color=CYAN, fontsize=9)
finscope_style(ax3, fig, 'Cluster Size & Avg Spending')
lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3_r.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2, fontsize=8,
           facecolor=PANEL, edgecolor=BORDER, labelcolor=WHITE)

plt.tight_layout(pad=2.5)
plt.savefig('finscope_kmeans_clusters.png', dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()

# Print cluster profiles
print('\n🔵 Cluster Profiles:')
CLUSTER_LABELS = ['💎 Power Spender', '🍔 Dining Lover', '🏠 Homebody Saver',
                  '🎮 Entertainer', '🚗 Commuter', '📚 Student', '💼 Professional', '🌿 Minimalist']
for c in range(K):
    top_cat = cluster_means.loc[c].idxmax() if c in cluster_means.index else 'N/A'
    members = (labels == c).sum()
    avg_total = cluster_summary.loc[c, 'avg_total'] if c in cluster_summary.index else 0
    label = CLUSTER_LABELS[c % len(CLUSTER_LABELS)]
    print(f'   Cluster {c+1} {label}: {members} members | Avg ${avg_total:,.0f} | Top: {top_cat}')

## 🌡️ 5. Heatmap & Correlation Matrix

In [ ]:
# ─── Fig 3: Advanced Heatmaps ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor=BG)
fig.suptitle('FinScope AI — Spending Heatmaps', color=WHITE, fontsize=15, fontweight='bold')

# ── Heatmap 1: Day × Category spending ──────────────────────────────────────
ax1 = axes[0]; ax1.set_facecolor(BG)
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot_heat = expenses.pivot_table(
    values='amount', index='day_of_week', columns='category',
    aggfunc='sum', fill_value=0
).reindex(day_order, fill_value=0)

sns.heatmap(
    pivot_heat, ax=ax1, cmap=FS_CMAP,
    linewidths=0.5, linecolor=BG,
    annot=True, fmt='.0f',
    annot_kws={'size': 8, 'color': WHITE, 'fontweight': 'bold'},
    cbar_kws={'shrink': 0.8, 'label': 'Amount ($)'}
)
ax1.set_title('Spending: Day × Category ($)', color=WHITE, fontsize=12, fontweight='bold')
ax1.set_xlabel('', color=TEXT); ax1.set_ylabel('', color=TEXT)
ax1.tick_params(colors=TEXT, labelsize=8)
ax1.collections[0].colorbar.ax.yaxis.set_tick_params(color=TEXT, labelcolor=TEXT)
ax1.collections[0].colorbar.set_label('Amount ($)', color=TEXT)

# ── Heatmap 2: Month × Category spending ────────────────────────────────────
ax2 = axes[1]; ax2.set_facecolor(BG)
expenses['month_str'] = expenses['month'].astype(str)
pivot_month_cat = expenses.pivot_table(
    values='amount', index='month_str', columns='category',
    aggfunc='sum', fill_value=0
)

sns.heatmap(
    pivot_month_cat, ax=ax2, cmap=FS_CMAP,
    linewidths=0.5, linecolor=BG,
    annot=True, fmt='.0f',
    annot_kws={'size': 8, 'color': WHITE, 'fontweight': 'bold'},
    cbar_kws={'shrink': 0.8}
)
ax2.set_title('Spending: Month × Category ($)', color=WHITE, fontsize=12, fontweight='bold')
ax2.set_xlabel('', color=TEXT); ax2.set_ylabel('', color=TEXT)
ax2.tick_params(colors=TEXT, labelsize=8, axis='x', rotation=45)
ax2.collections[0].colorbar.ax.yaxis.set_tick_params(color=TEXT, labelcolor=TEXT)

plt.tight_layout(pad=2.5)
plt.savefig('finscope_heatmaps.png', dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()
print('✅ Saved: finscope_heatmaps.png')

In [ ]:
# ─── Correlation Matrix ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 9), facecolor=BG)
ax.set_facecolor(BG)

# Build corr matrix from monthly per-category spending
corr_pivot = expenses.groupby(['month_str', 'category'])['amount'].sum().unstack(fill_value=0)
corr_matrix = corr_pivot.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap_div = LinearSegmentedColormap.from_list('diverg', [RED, BG, CYAN], N=256)

sns.heatmap(
    corr_matrix, ax=ax, cmap=cmap_div,
    mask=mask, vmin=-1, vmax=1, center=0,
    linewidths=1, linecolor=BG,
    annot=True, fmt='.2f',
    annot_kws={'size': 9, 'fontweight': 'bold'},
    square=True,
    cbar_kws={'shrink': 0.8, 'label': 'Pearson r'}
)
ax.set_title('Category Spending Correlation Matrix\n(Do certain categories move together?)',
             color=WHITE, fontsize=13, fontweight='bold', pad=14)
ax.tick_params(colors=TEXT, labelsize=10)
ax.collections[0].colorbar.ax.yaxis.set_tick_params(color=TEXT, labelcolor=TEXT)
ax.collections[0].colorbar.set_label('Pearson r', color=TEXT)

plt.tight_layout()
plt.savefig('finscope_correlation.png', dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()

# Print strongest correlations
corr_unstacked = corr_matrix.unstack()
corr_filtered = corr_unstacked[(corr_unstacked < 0.99) & (corr_unstacked > 0)].sort_values(ascending=False)
print('\n📊 Strongest Category Correlations:')
for (cat1, cat2), val in corr_filtered.head(5).items():
    print(f'   {cat1} ↔ {cat2}: r = {val:.3f}')

## 📈 6. Time-Series Decomposition & Trends

In [ ]:
# ─── Fig 5: Advanced Time-Series Analysis ────────────────────────────────────
# Daily spending time-series
daily = expenses.groupby('date')['amount'].sum().reset_index()
daily['date'] = pd.to_datetime(daily['date'])
daily = daily.sort_values('date').set_index('date')

# Rolling stats
daily['rolling_7']  = daily['amount'].rolling(7, min_periods=1).mean()
daily['rolling_30'] = daily['amount'].rolling(30, min_periods=1).mean()
daily['cumsum']     = daily['amount'].cumsum()
daily['pct_change'] = daily['amount'].pct_change().fillna(0) * 100

# ── Linear trend ─────────────────────────────────────────────────────────────
x_num = np.arange(len(daily))
lr = LinearRegression()
lr.fit(x_num.reshape(-1,1), daily['amount'].values)
trend_line = lr.predict(x_num.reshape(-1,1))
trend_dir  = '📈 Increasing' if lr.coef_[0] > 0 else '📉 Decreasing'

fig = plt.figure(figsize=(18, 12), facecolor=BG)
fig.suptitle('FinScope AI — Time-Series Decomposition', color=WHITE, fontsize=15, fontweight='bold')
gs2 = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.32,
                        top=0.93, bottom=0.06, left=0.07, right=0.97)

# ── A: Daily + Rolling averages ───────────────────────────────────────────
ax_ts = fig.add_subplot(gs2[0, :]); ax_ts.set_facecolor(BG)
ax_ts.fill_between(daily.index, daily['amount'], alpha=0.12, color=BLUE)
ax_ts.plot(daily.index, daily['amount'], color=BLUE + '66', linewidth=0.8, label='Daily')
ax_ts.plot(daily.index, daily['rolling_7'],  color=CYAN,   linewidth=2.2, label='7-day MA')
ax_ts.plot(daily.index, daily['rolling_30'], color=PURPLE, linewidth=2.0, linestyle='--', label='30-day MA')
ax_ts.plot(daily.index, trend_line, color=YELLOW, linewidth=1.5, linestyle=':', alpha=0.85, label=f'Trend ({trend_dir})')
ax_ts.set_ylabel('Daily Spending ($)', color=TEXT, fontsize=9)
finscope_style(ax_ts, fig, 'Daily Spending + Moving Averages')
ax_ts.legend(fontsize=9, facecolor=PANEL, edgecolor=BORDER, labelcolor=WHITE)
ax_ts.tick_params(axis='x', colors=TEXT, labelsize=8)

# ── B: Cumulative spending ────────────────────────────────────────────────
ax_cum = fig.add_subplot(gs2[1, 0]); ax_cum.set_facecolor(BG)
ax_cum.fill_between(daily.index, daily['cumsum'], alpha=0.25, color=CYAN)
ax_cum.plot(daily.index, daily['cumsum'], color=CYAN, linewidth=2.2)
ax_cum.set_ylabel('Cumulative Spend ($)', color=TEXT, fontsize=9)
finscope_style(ax_cum, fig, 'Cumulative Spending Over Time')

# ── C: Day-over-day % change ──────────────────────────────────────────────
ax_pct = fig.add_subplot(gs2[1, 1]); ax_pct.set_facecolor(BG)
pos = daily['pct_change'] >= 0
ax_pct.bar(daily.index[pos],  daily['pct_change'][pos],  color=RED,   width=0.8, edgecolor='none', alpha=0.85)
ax_pct.bar(daily.index[~pos], daily['pct_change'][~pos], color=GREEN, width=0.8, edgecolor='none', alpha=0.85)
ax_pct.axhline(0, color=BORDER, linewidth=1)
ax_pct.set_ylabel('Day-over-Day Change (%)', color=TEXT, fontsize=9)
finscope_style(ax_pct, fig, 'Day-over-Day Spending Change')

# ── D: Monthly bar with deposits vs expenses ──────────────────────────────
ax_bar2 = fig.add_subplot(gs2[2, 0]); ax_bar2.set_facecolor(BG)
monthly_exp = expenses.groupby('month')['amount'].sum().reset_index()
monthly_dep = deposits.groupby('month')['amount'].sum().reset_index()
months_all = sorted(set(list(monthly_exp['month']) + list(monthly_dep['month'])))
months_str = [str(m) for m in months_all]
exp_vals = [monthly_exp[monthly_exp['month'] == m]['amount'].values[0]
            if m in monthly_exp['month'].values else 0 for m in months_all]
dep_vals = [monthly_dep[monthly_dep['month'] == m]['amount'].values[0]
            if m in monthly_dep['month'].values else 0 for m in months_all]
xpos = np.arange(len(months_all))
w2 = 0.38
ax_bar2.bar(xpos - w2/2, dep_vals, w2, color=GREEN + 'bb', edgecolor='none', label='Income')
ax_bar2.bar(xpos + w2/2, exp_vals, w2, color=RED   + 'bb', edgecolor='none', label='Expenses')
ax_bar2.set_xticks(xpos)
ax_bar2.set_xticklabels(months_str, rotation=35, ha='right', color=TEXT, fontsize=8)
ax_bar2.set_ylabel('Amount ($)', color=TEXT, fontsize=9)
finscope_style(ax_bar2, fig, 'Income vs Expenses by Month')
ax_bar2.legend(fontsize=9, facecolor=PANEL, edgecolor=BORDER, labelcolor=WHITE)

# ── E: Savings rate waterfall ─────────────────────────────────────────────
ax_save = fig.add_subplot(gs2[2, 1]); ax_save.set_facecolor(BG)
savings = [d - e for d, e in zip(dep_vals, exp_vals)]
sav_colors = [GREEN if s >= 0 else RED for s in savings]
ax_save.bar(xpos, savings, color=sav_colors, edgecolor='none', width=0.65, alpha=0.85)
ax_save.axhline(0, color=BORDER, linewidth=1)
for i, (v, c) in enumerate(zip(savings, sav_colors)):
    ax_save.text(i, v + (max(savings) * 0.03 if v >= 0 else min(savings) * 0.03),
                 f'${v:,.0f}', ha='center', color=WHITE, fontsize=8, fontweight='bold')
ax_save.set_xticks(xpos)
ax_save.set_xticklabels(months_str, rotation=35, ha='right', color=TEXT, fontsize=8)
ax_save.set_ylabel('Net Savings ($)', color=TEXT, fontsize=9)
finscope_style(ax_save, fig, 'Monthly Net Savings')

plt.savefig('finscope_timeseries.png', dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'✅ Saved: finscope_timeseries.png')
print(f'   Trend: {trend_dir} (slope = ${lr.coef_[0]:.2f}/day)')

## 🤖 7. Anomaly Detection

In [ ]:
# ─── Isolation Forest: Multi-feature anomaly detection ───────────────────────
print('🚨 Running Anomaly Detection (Isolation Forest)...')

features_df = expenses[['amount', 'day_num', 'hour', 'day_of_month']].copy()
scaler_anom = StandardScaler()
X_anom = scaler_anom.fit_transform(features_df)

iso = IsolationForest(contamination=0.04, random_state=42, n_estimators=200)
preds  = iso.fit_predict(X_anom)
scores = iso.score_samples(X_anom)

expenses_flagged = expenses.copy()
expenses_flagged['is_anomaly']    = preds == -1
expenses_flagged['anomaly_score'] = scores

normal   = expenses_flagged[~expenses_flagged['is_anomaly']]
anomalies = expenses_flagged[expenses_flagged['is_anomaly']].sort_values('anomaly_score')

print(f'   Found {len(anomalies)} anomalies out of {len(expenses)} expenses ({len(anomalies)/len(expenses)*100:.1f}%)')

# ── Visualize anomalies ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor=BG)
fig.suptitle('FinScope AI — Anomaly Detection', color=WHITE, fontsize=14, fontweight='bold')

# Panel 1: Scatter normal vs anomaly
ax1 = axes[0]; ax1.set_facecolor(BG)
ax1.scatter(normal.index, normal['amount'], color=BLUE, alpha=0.5, s=25, label='Normal', edgecolors='none')
ax1.scatter(anomalies.index, anomalies['amount'], color=RED, s=90, zorder=5,
            edgecolors=WHITE, linewidths=0.8, label=f'Anomaly 🚨 ({len(anomalies)})')
for _, row in anomalies.head(8).iterrows():
    ax1.annotate(
        f"{row['name'][:12]}\n${row['amount']:.0f}",
        (row.name, row['amount']),
        textcoords='offset points', xytext=(8, 4),
        color='#fca5a5', fontsize=7, fontweight='bold'
    )
ax1.set_ylabel('Amount ($)', color=TEXT, fontsize=9)
finscope_style(ax1, fig, 'Transaction Anomaly Scatter')
ax1.legend(fontsize=9, facecolor=PANEL, edgecolor=BORDER, labelcolor=WHITE)
ax1.tick_params(axis='x', labelbottom=False)

# Panel 2: Anomaly score distribution
ax2 = axes[1]; ax2.set_facecolor(BG)
ax2.hist(scores[preds == 1],  bins=30, color=BLUE,   alpha=0.7, label='Normal',   edgecolor='none')
ax2.hist(scores[preds == -1], bins=10, color=RED,    alpha=0.85, label='Anomaly',  edgecolor='none')
ax2.axvline(scores.mean(), color=YELLOW, linestyle='--', linewidth=1.5, label='Mean score')
ax2.set_xlabel('Anomaly Score (lower = more suspicious)', color=TEXT, fontsize=9)
ax2.set_ylabel('Count', color=TEXT, fontsize=9)
finscope_style(ax2, fig, 'Anomaly Score Distribution')
ax2.legend(fontsize=9, facecolor=PANEL, edgecolor=BORDER, labelcolor=WHITE)

plt.tight_layout(pad=3)
plt.savefig('finscope_anomalies.png', dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()

print('\n⚠️  Top 10 Most Suspicious Transactions:')
display(
    anomalies[['created_at','name','category','amount','anomaly_score']]
    .head(10)
    .rename(columns={'created_at':'Date','name':'Merchant',
                     'category':'Category','amount':'Amount ($)','anomaly_score':'Score'})
    .style
    .background_gradient(subset=['Amount ($)'], cmap='Reds')
    .format({'Amount ($)': '${:.2f}', 'Score': '{:.4f}'})
)

## 🔮 8. Spending Forecast (Linear + Polynomial)

In [ ]:
# ─── Forecast next-month spending ────────────────────────────────────────────
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

monthly_totals = expenses.groupby('month')['amount'].sum().reset_index()
monthly_totals = monthly_totals.sort_values('month')
monthly_totals['month_str'] = monthly_totals['month'].astype(str)

N = len(monthly_totals)
X_m = np.arange(N).reshape(-1, 1)
y_m = monthly_totals['amount'].values

# Linear
lr_m = LinearRegression(); lr_m.fit(X_m, y_m)
# Polynomial degree-2
poly_m = make_pipeline(PolynomialFeatures(degree=2), LinearRegression())
poly_m.fit(X_m, y_m)

# Project 3 months ahead
HORIZON = 3
X_fut = np.arange(N, N + HORIZON).reshape(-1, 1)
pred_lin  = lr_m.predict(X_fut)
pred_poly = poly_m.predict(X_fut)

# Confidence interval (2× residual std)
resid_std = np.std(y_m - lr_m.predict(X_m))

future_labels = [str((pd.Period(monthly_totals['month_str'].iloc[-1]) + i + 1))
                 for i in range(HORIZON)]

fig, ax = plt.subplots(figsize=(14, 6), facecolor=BG)
ax.set_facecolor(BG)

# Historical
x_hist = list(range(N))
ax.fill_between(x_hist, y_m, alpha=0.15, color=BLUE)
ax.plot(x_hist, y_m, color=BLUE, linewidth=2.5, marker='o',
        markersize=8, markerfacecolor=WHITE, markeredgecolor=BLUE, markeredgewidth=2, label='Actual')

# Linear forecast
ax.plot([N-1] + list(range(N, N+HORIZON)),
        [y_m[-1]] + list(pred_lin),
        color=CYAN, linewidth=2, linestyle='--', marker='o',
        markersize=7, markerfacecolor=CYAN, label='Linear Forecast')
ax.fill_between(range(N-1, N+HORIZON),
                [y_m[-1]] + list(pred_lin - 2*resid_std),
                [y_m[-1]] + list(pred_lin + 2*resid_std),
                alpha=0.12, color=CYAN)

# Polynomial forecast
ax.plot([N-1] + list(range(N, N+HORIZON)),
        [y_m[-1]] + list(pred_poly),
        color=PURPLE, linewidth=2, linestyle='-.', marker='s',
        markersize=6, markerfacecolor=PURPLE, label='Poly-2 Forecast')

# Annotations
for i, (pl, pp) in enumerate(zip(pred_lin, pred_poly)):
    ax.annotate(f'${pl:,.0f}', (N + i, pl),
                textcoords='offset points', xytext=(4, 10),
                color=CYAN, fontsize=9, fontweight='bold')

# X-axis labels
all_labels = list(monthly_totals['month_str']) + future_labels
ax.set_xticks(range(len(all_labels)))
ax.set_xticklabels(all_labels, rotation=35, ha='right', color=TEXT, fontsize=9)
ax.axvline(N - 0.5, color=YELLOW, linestyle=':', linewidth=1.5, alpha=0.7)
ax.text(N - 0.4, ax.get_ylim()[1] * 0.97, '← History | Forecast →',
        color=YELLOW, fontsize=9, va='top')
ax.set_ylabel('Monthly Spending ($)', color=TEXT, fontsize=9)
finscope_style(ax, fig, 'Monthly Spending Forecast (3 Months Ahead)')
ax.legend(fontsize=10, facecolor=PANEL, edgecolor=BORDER, labelcolor=WHITE)

plt.tight_layout()
plt.savefig('finscope_forecast.png', dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()

r2 = lr_m.score(X_m, y_m)
trend = '📈 Increasing' if lr_m.coef_[0] > 0 else '📉 Decreasing'
print(f'\n🔮 Forecast Summary:')
print(f'   Model R²  : {r2:.3f}')
print(f'   Trend     : {trend} (+${lr_m.coef_[0]:,.2f}/month)')
for label, val in zip(future_labels, pred_lin):
    lo, hi = val - 2*resid_std, val + 2*resid_std
    print(f'   {label} : ${max(0,val):,.2f}  (range ${max(0,lo):,.2f}–${hi:,.2f})')

## 🧬 9. Spending DNA — User Behavior Fingerprint

In [ ]:
# ─── Radar / Spider Chart: User Spending DNA ─────────────────────────────────
from matplotlib.patches import FancyBboxPatch

users = expenses['user_id'].unique()[:4]  # Max 4 for readability
cats_for_radar = [c for c in expenses['category'].unique()
                  if expenses[expenses['category'] == c]['amount'].sum() > 0]
cats_for_radar = cats_for_radar[:8]  # Limit to 8 categories
N_cat = len(cats_for_radar)

# Compute per-user category percentages
def user_radar_data(uid):
    udf = expenses[expenses['user_id'] == uid]
    total = udf['amount'].sum()
    if total == 0: return [0] * N_cat
    return [(udf[udf['category'] == c]['amount'].sum() / total) * 100 for c in cats_for_radar]

angles = np.linspace(0, 2 * np.pi, N_cat, endpoint=False).tolist()
angles += angles[:1]

n_users = len(users)
ncols = min(n_users, 2)
nrows = (n_users + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, subplot_kw=dict(polar=True),
                         figsize=(7 * ncols, 6 * nrows), facecolor=BG)
if n_users == 1:
    axes = np.array([axes])
axes = np.array(axes).flatten()

radar_colors = [BLUE, CYAN, PURPLE, PINK]
CLUSTER_LABELS_SHORT = ['💎 Power Spender', '🍔 Dining Lover', '🏠 Saver', '🎮 Entertainer']

for idx, (uid, ax) in enumerate(zip(users, axes)):
    ax.set_facecolor(BG)
    ax.spines['polar'].set_color(BORDER)
    ax.yaxis.set_tick_params(labelcolor=TEXT, labelsize=7)
    ax.xaxis.set_tick_params(labelcolor=TEXT)
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    values = user_radar_data(uid)
    values += values[:1]

    col = radar_colors[idx % len(radar_colors)]
    ax.plot(angles, values, color=col, linewidth=2.5)
    ax.fill(angles, values, color=col, alpha=0.2)

    # Dots at each vertex
    ax.scatter(angles[:-1], values[:-1], color=col, s=60,
               zorder=5, edgecolors=WHITE, linewidths=1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(cats_for_radar, color=TEXT, fontsize=9, fontweight='bold')
    ax.set_ylim(0, max(values) * 1.25 + 5)
    ax.yaxis.set_major_locator(plt.MaxNLocator(4))
    ax.grid(color=BORDER, linewidth=0.7, alpha=0.6)

    label = CLUSTER_LABELS_SHORT[idx % len(CLUSTER_LABELS_SHORT)]
    uid_short = str(uid)[:12] + ('…' if len(str(uid)) > 12 else '')
    ax.set_title(f'{uid_short}\n{label}', color=WHITE, fontsize=11, fontweight='bold', pad=18)

# Hide unused axes
for ax in axes[n_users:]:
    ax.set_visible(False)

fig.suptitle('FinScope AI — Spending DNA (Radar Chart)', color=WHITE,
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(pad=3)
plt.savefig('finscope_spending_dna.png', dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()
print('✅ Saved: finscope_spending_dna.png')

In [ ]:
# ─── Bubble Chart: Category × Risk × Volume ──────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 8), facecolor=BG)
ax.set_facecolor(BG)

cat_stats = expenses.groupby('category').agg(
    total=('amount', 'sum'),
    count=('amount', 'count'),
    avg=('amount', 'mean'),
    std=('amount', 'std'),
    anomaly_rate=('amount', lambda x: (x > x.mean() + 2*x.std()).mean())
).fillna(0)

max_total = cat_stats['total'].max()
for cat, row in cat_stats.iterrows():
    col = CAT_COLORS.get(cat, GREY)
    size = (row['total'] / max_total) * 3000 + 100
    ax.scatter(row['count'], row['avg'], s=size, color=col,
               alpha=0.75, edgecolors=WHITE, linewidths=1)
    ax.annotate(
        f"{cat}\n${row['total']:,.0f}",
        (row['count'], row['avg']),
        textcoords='offset points', xytext=(8, 5),
        color=WHITE, fontsize=9, fontweight='bold'
    )

ax.set_xlabel('Number of Transactions', color=TEXT, fontsize=10)
ax.set_ylabel('Average Transaction ($)', color=TEXT, fontsize=10)
finscope_style(ax, fig, 'Category Bubble Chart',
               subtitle='Bubble size = Total spend | X = Frequency | Y = Avg amount')

plt.tight_layout()
plt.savefig('finscope_bubble_chart.png', dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()
print('✅ Saved: finscope_bubble_chart.png')

## 💾 10. Export Summary Report

In [ ]:
# ─── Export comprehensive JSON report ────────────────────────────────────────
import json

def make_serializable(obj):
    if isinstance(obj, (pd.Period, pd.Timestamp)): return str(obj)
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, np.bool_): return bool(obj)
    if isinstance(obj, dict): return {k: make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [make_serializable(i) for i in obj]
    return obj

report = {
    'generated_at': datetime.now().isoformat(),
    'dataset': {
        'total_transactions': len(df),
        'total_expenses': len(expenses),
        'total_deposits': len(deposits),
        'unique_users': int(df['user_id'].nunique()),
        'date_range': {
            'from': str(df['created_at'].min().date()),
            'to': str(df['created_at'].max().date())
        },
        'total_spent': float(expenses['amount'].sum()),
        'total_income': float(deposits['amount'].sum()),
    },
    'category_breakdown': make_serializable(
        expenses.groupby('category')['amount'].agg(['sum','count','mean']).to_dict()
    ),
    'kmeans': {
        'optimal_k': int(best_k),
        'silhouette_score': float(max(sils)),
        'cluster_profiles': make_serializable(
            pivot.groupby('cluster')['total'].agg(['mean','count']).to_dict()
        ) if 'total' in pivot.columns else {}
    },
    'anomalies': {
        'count': int(len(anomalies)),
        'rate_pct': float(len(anomalies) / len(expenses) * 100),
        'top_anomalies': make_serializable(
            anomalies[['name','category','amount']].head(5).to_dict('records')
        )
    },
    'forecast': {
        'model_r2': float(r2),
        'trend': trend,
        'monthly_slope': float(lr_m.coef_[0]),
        'next_3_months': [
            {'month': label, 'predicted': float(max(0, val)),
             'low': float(max(0, val - 2*resid_std)), 'high': float(val + 2*resid_std)}
            for label, val in zip(future_labels, pred_lin)
        ]
    },
    'charts_generated': [
        'finscope_dashboard_overview.png',
        'finscope_kmeans_selection.png',
        'finscope_kmeans_clusters.png',
        'finscope_heatmaps.png',
        'finscope_correlation.png',
        'finscope_timeseries.png',
        'finscope_anomalies.png',
        'finscope_forecast.png',
        'finscope_spending_dna.png',
        'finscope_bubble_chart.png',
    ]
}

with open('finscope_full_report.json', 'w') as f:
    json.dump(make_serializable(report), f, indent=2)

print('╔══════════════════════════════════════════════════════╗')
print('║     FinScope AI — Analysis Complete! ✅              ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  📊 Transactions analysed : {report["dataset"]["total_transactions"]:>6,}                 ║')
print(f'║  💸 Total spent           : ${report["dataset"]["total_spent"]:>10,.2f}             ║')
print(f'║  🔵 K-Means clusters      : K={best_k}  (sil={max(sils):.3f})        ║')
print(f'║  🚨 Anomalies detected    : {report["anomalies"]["count"]:>4} ({report["anomalies"]["rate_pct"]:.1f}%)              ║')
print(f'║  🔮 Forecast trend        : {trend:>20}        ║')
print(f'║  🖼️  Charts saved          : {len(report["charts_generated"]):>4}                         ║')
print('╚══════════════════════════════════════════════════════╝')
print()
print('  📄 Report saved to: finscope_full_report.json')